# 02. 데이터 파이프라인과 MLP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2038prob/defense-ai-textbook-code/blob/main/notebooks/%EC%8B%A4%EC%8A%B5_02_%EB%8D%B0%EC%9D%B4%ED%84%B0_%ED%8C%8C%EC%9D%B4%ED%94%84%EB%9D%BC%EC%9D%B8%EA%B3%BC_MLP.ipynb)

「국방 인공지능」 실습 코드 — 이 장의 파이썬 코드를 순서대로 모았습니다. **맨 위 setup 셀부터 위에서 아래로** 실행하세요(앞 셀에서 정의한 변수를 뒤 셀이 이어 씁니다). ⚠️ 표시가 붙은 셀은 개념 설명용 조각으로, 그대로는 실행되지 않을 수 있습니다.

전체 저장소: https://github.com/2038prob/defense-ai-textbook-code


In [ ]:
# === 실습 환경 준비 (Colab/로컬 공통) — 가장 먼저 실행하세요 ===
# 필요한 패키지를 설치하고(이미 있으면 건너뜀) 공통 유틸을 정의합니다.
import importlib.util, subprocess, sys
for pkg in ["torch", "torchvision", "numpy", "matplotlib", "scipy"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

## 역전파(Backpropagation) 유도

손으로 구한 위 기울기가 정말 맞는지 PyTorch로 검산해 볼 수 있다. 아래 코드를 돌리면 \( L/ w_1=-2,\ L/ w_2=-0.5\)가 그대로 나온다.


In [ ]:
import torch
x = torch.tensor([1.0])
w1 = torch.tensor([0.5], requires_grad=True)
b1 = torch.tensor([0.0], requires_grad=True)
w2 = torch.tensor([2.0], requires_grad=True)
b2 = torch.tensor([0.0], requires_grad=True)
t  = torch.tensor([2.0])

a1 = w1 * x + b1
h  = torch.relu(a1)            # ReLU
a2 = w2 * h + b2
y_hat = a2                     # 선형 출력
loss = 0.5 * (y_hat - t) ** 2

loss.backward()                # 자동 미분
print(w1.grad, w2.grad)        # tensor([-2.]) tensor([-0.5000])

## L2 정규화·Dropout·Batch Normalization


In [ ]:
import torch.nn as nn

class RegularizedMLP(nn.Module):
    """Dropout + BatchNorm을 포함한 MLP"""
    def __init__(self, input_dim, hidden_dim, output_dim, dropout_p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),     # 배치 정규화
            nn.ReLU(),
            nn.Dropout(p=dropout_p),        # Dropout (학습 시만 작동)
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(hidden_dim, output_dim),
        )
    def forward(self, x):
        return self.net(x)

## 가중치 초기화


In [ ]:
def initialize_weights(model):
    """He 초기화 (ReLU 기반 네트워크 권장)"""
    for module in model.modules():
        if isinstance(module, nn.Linear):
            nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='relu')
            if module.bias is not None:
                nn.init.zeros_(module.bias)

## 데이터 파이프라인: Dataset과 DataLoader


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split

class EquipmentDataset(Dataset):
    """합성 센서 데이터셋"""
    def __init__(self, n=1000, input_dim=4, num_classes=3, seed=0):
        g = torch.Generator().manual_seed(seed)
        self.x = torch.randn(n, input_dim, generator=g)
        self.y = torch.randint(0, num_classes, (n,), generator=g)
    def __len__(self):
        return len(self.x)
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

full = EquipmentDataset(n=2000)
train_size = int(0.8 * len(full)); val_size = len(full) - train_size
train_set, val_set = random_split(full, [train_size, val_size])
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)   # 학습: 셔플
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False) # 검증: 고정

In [ ]:
from torch.utils.data import WeightedRandomSampler

# 클래스 빈도의 역수를 샘플 가중치로 -> 희귀 클래스가 더 자주 뽑힘
labels = full.y
class_count = torch.bincount(labels)                 # 각 클래스 개수
class_weight = 1.0 / class_count.float()             # 빈도의 역수
sample_weight = class_weight[labels]                 # 샘플별 가중치
sampler = WeightedRandomSampler(sample_weight, num_samples=len(labels),
                                replacement=True)
balanced_loader = DataLoader(full, batch_size=64, sampler=sampler)
# 주의: sampler를 쓰면 shuffle은 지정하지 않는다(둘은 함께 못 씀).

앞의 `EquipmentDataset`은 설명을 위해 난수로 데이터를 모사했지만, 실제 프로젝트에서는 보통 CSV 파일 같은 원본에서 데이터를 읽어 텐서로 바꾼다. 그 전형적인 모습은 다음과 같다.


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset

class EquipmentCSVDataset(Dataset):
    def __init__(self, csv_path):
        df = pd.read_csv(csv_path)                  # 원본 읽기
        feats = df[["temp", "vibration", "pressure", "current"]].values
        labels = df["status"].values               # 0/1/2
        self.x = torch.tensor(feats, dtype=torch.float32)
        self.y = torch.tensor(labels, dtype=torch.long)   # 분류 레이블은 long
    def __len__(self):
        return len(self.x)
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

## 모델 정의와 학습 루프


In [ ]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

### 🔬 실습 — 장비 분류 MLP 학습 (조기 종료 포함)

① **설정** — 손실함수, optimizer, 학습률 스케줄러, 조기 종료 변수.


In [ ]:
import torch, torch.nn as nn, torch.optim as optim
torch.manual_seed(0)
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = RegularizedMLP(4, 64, 3, dropout_p=0.3).to(device)
initialize_weights(model)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                 patience=5, factor=0.5)
PATIENCE = 10; best_val = float('inf'); best_state = None; cnt = 0
def accuracy(logits, y): return (logits.argmax(1) == y).float().mean().item()

② **학습 단계** — 매 배치마다 zero_grad $$ backward $$ step.


In [ ]:
for epoch in range(1, 101):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = criterion(model(x), y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    model.eval(); va_loss = va_acc = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            va_loss += criterion(logits, y).item()
            va_acc  += accuracy(logits, y)
    va_loss /= len(val_loader); va_acc /= len(val_loader)
    scheduler.step(va_loss)
    print(f"Ep {epoch:03d} | val {va_loss:.4f}/{va_acc:.3f}")
    if va_loss < best_val - 1e-4:
        best_val = va_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        cnt = 0
    else:
        cnt += 1
        if cnt >= PATIENCE:
            print(f"조기 종료: {epoch}에포크 (최적 검증 손실 {best_val:.4f})")
            break
model.load_state_dict(best_state)

③ **검증 단계** — `eval()`로 전환, `no_grad()`로 평가.


④ **조기 종료** — 검증 손실이 개선 없으면 중단하고 최적 가중치 복원.


### 🔬 실습 — 과적합을 일부러 일으키고, 눈으로 진단하고, 고쳐 보기

① **과적합이 잘 일어나는 조건 만들기**: 데이터는 적게(300개), 모델은 크게(4층$$512).


In [ ]:
import torch, torch.nn as nn

torch.manual_seed(42)
# 작은 합성 데이터: 3클래스 장비 상태, 특성 20개 중 유효한 것은 5개뿐
X = torch.randn(300, 20)
w_true = torch.zeros(20, 3); w_true[:5] = torch.randn(5, 3) * 2.0
y = (X @ w_true + 0.8 * torch.randn(300, 3)).argmax(1)   # 레이블 잡음 포함
X_tr, y_tr, X_va, y_va = X[:200], y[:200], X[200:], y[200:]

def make_mlp(p_drop=0.0):
    return nn.Sequential(
        nn.Linear(20, 512), nn.ReLU(), nn.Dropout(p_drop),
        nn.Linear(512, 512), nn.ReLU(), nn.Dropout(p_drop),
        nn.Linear(512, 512), nn.ReLU(), nn.Dropout(p_drop),
        nn.Linear(512, 3))

② **두 설정을 같은 데이터로 학습**: (가) 정규화 없음, (나) Dropout 0.5 + weight decay.


In [ ]:
def run(model, weight_decay=0.0, epochs=300):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3,
                           weight_decay=weight_decay)
    crit = nn.CrossEntropyLoss()
    hist = []
    for ep in range(epochs):
        model.train()
        loss = crit(model(X_tr), y_tr)
        opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            va_loss = crit(model(X_va), y_va).item()
            tr_acc = (model(X_tr).argmax(1) == y_tr).float().mean().item()
            va_acc = (model(X_va).argmax(1) == y_va).float().mean().item()
        hist.append((loss.item(), va_loss, tr_acc, va_acc))
    return hist

hist_over = run(make_mlp(0.0), weight_decay=0.0)   # (가) 과적합 유도
hist_reg  = run(make_mlp(0.5), weight_decay=1e-3)  # (나) 정규화 적용
for name, h in [("no-reg", hist_over), ("reg", hist_reg)]:
    tr_l, va_l, tr_a, va_a = h[-1]
    print(f"{name:6s} | train acc {tr_a:.3f} | val acc {va_a:.3f} "
          f"| val loss {va_l:.3f}")

**실행 결과 예시**

```
no-reg | train acc 1.000 | val acc 0.620 | val loss 2.417 \\
reg    | train acc 0.845 | val acc 0.740 | val loss 0.688
```


---

실습 코드 저장소: https://github.com/2038prob/defense-ai-textbook-code  ·  교재 「국방 인공지능」
